<a href="https://colab.research.google.com/github/41371103hjnh/114-1-/blob/main/HW6_%E8%AA%B2%E7%A8%8B%E5%B0%8F%E5%8A%A9%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# 0. 安裝套件（只在第一次跑）
# =========================================================
!pip install -q google-generativeai gspread gspread_dataframe gradio pandas
!pip install -q feedparser
!pip install -q requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.2 MB/s eta 0:00:00


In [2]:
# =========================================================
# 1. 匯入與基本設定：Gemini + Google Sheet + OpenWeather
# =========================================================
import os
import io
import time
import json
import traceback
import datetime as dt

import pandas as pd
import google.generativeai as genai

from google.colab import auth, files
from google.auth import default
import gspread
from gspread_dataframe import set_with_dataframe

import requests
import feedparser
import re
import gradio as gr

# ---------- 1-1. 讀取 Gemini API Key ----------
GEMINI_KEY = None
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get("hw6")
    if GEMINI_KEY:
        print("🔐 已從 Colab Secret 'hw6' 讀取 Gemini API Key")
except Exception as e:
    print("⚠️ 讀取 Colab userdata 失敗，準備使用備用 key 或手動輸入")

# 如果 Secret 沒有，就用寫死的 key（請自行填入）
if not GEMINI_KEY:
    GEMINI_KEY = "👉在這裡放你的 Gemini API Key👈"
    print("⚠️ 未找到 Secret 'hw6'，改用程式裡寫死的 API Key")

# ---------- 1-2. 設定 Gemini ----------
genai.configure(api_key=GEMINI_KEY)

MODEL_NAME = "gemini-2.5-flash"
model = genai.GenerativeModel(MODEL_NAME)

print(f"✅ Gemini 已就緒，使用模型：{MODEL_NAME}")

# ---------- 1-3. Google 授權 + 試算表連線 ----------
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1f0Xha99x0pBBmtJDqq_dFR7c-O1H6afiDgjYQwo5QWE/edit?usp=sharing"
WORKSHEET_NAME = "課表"

try:
    sh = gc.open_by_url(SHEET_URL)
    ws = sh.worksheet(WORKSHEET_NAME)
    print(f"✅ 已連線到試算表：{sh.title} → 工作表：{WORKSHEET_NAME}")
except Exception as e:
    print("❌ 找不到試算表或工作表，請確認 URL 或名稱是否正確")
    raise e

# ---------- 1-4. OpenWeather API 設定（天氣） ----------
OPENWEATHER_API_KEY = "0b97ecd77aa2ceb4322c478706c3188d"  # ← 可以改成你的 key


def get_current_weather(city: str = "Taipei", country_code: str = "TW", lang: str = "zh_tw"):
    """
    呼叫 OpenWeather Current Weather API
    回傳一個簡單的 dict：{description, temp, feels_like}
    """
    base_url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": f"{city},{country_code}",   # 例如 Taipei,TW
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",               # 攝氏
        "lang": lang,                    # 回傳中文描述 zh_tw
    }

    resp = requests.get(base_url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    weather_desc = data["weather"][0]["description"]  # 例如 "小雨"
    temp = data["main"]["temp"]                       # 現在溫度
    feels_like = data["main"]["feels_like"]

    return {
        "description": weather_desc,
        "temp": temp,
        "feels_like": feels_like,
    }


def build_weather_note(city: str = "Taipei", country_code: str = "TW") -> str:
    """
    把天氣資訊組成一小段「今日天氣」文字。
    若 API 出錯，回傳空字串，不影響主流程。
    """
    try:
        w = get_current_weather(city=city, country_code=country_code, lang="zh_tw")
    except Exception as e:
        print("⚠️ 取得天氣失敗：", e)
        return ""  # 不影響主流程

    desc = w["description"]
    temp = round(w["temp"])
    feels = round(w["feels_like"])

    note_lines = [
        f"🌤 即時天氣：{desc}，約 {temp}°C，體感 {feels}°C。",
    ]

    # 簡單溫度提醒
    if temp >= 30:
        note_lines.append("天氣偏熱，上下課記得多補充水分。")
    elif temp <= 18:
        note_lines.append("氣溫較低，出門可以帶件外套保暖。")
    else:
        note_lines.append("溫度舒適，輕便服裝即可。")

    return "\n".join(note_lines)


🔐 已從 Colab Secret 'hw6' 讀取 Gemini API Key
✅ Gemini 已就緒，使用模型：gemini-2.5-flash
✅ 已連線到試算表：程式語言hw6 → 工作表：課表


In [4]:
# =========================================================
# 2. PDF ➜ CSV：使用 Gemini 解析課表
# =========================================================
def pdf_path_to_csv_text(pdf_path: str) -> str:
    """
    1. 將 pdf_path 上傳到 Gemini File API
    2. 讓 Gemini 依指定欄位輸出 CSV 文字
    3. 將結果存成 gemini_schedule.csv
    4. 回傳 CSV 字串
    """

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"找不到檔案：{pdf_path}")

    print(f"準備處理檔案: '{pdf_path}'")

    # 1. 上傳到 Gemini File API
    print("正在上傳 PDF 至 Gemini File API...")
    gemini_file = genai.upload_file(
        path=pdf_path,
        display_name="114-1 課程表 PDF"
    )
    print(f"✅ 檔案上傳成功！(File URI: {gemini_file.uri})")

    # 2. Prompt：請留「當日提醒」「學習建議」欄位空白
    prompt_text = f"""
請分析我上傳的 PDF 課表檔案 (檔案名稱: {os.path.basename(pdf_path)})。
請將 PDF 課表內容轉成符合下列格式的 CSV：

欄位順序固定：
日期,星期,課程名稱,時間(起),時間(迄),地點,當日提醒,學習建議

請嚴格遵守：
- 僅輸出 CSV（不能有其他說明文字）
- 日期格式：YYYY/MM/DD
- 星期：星期一〜星期日
- 時間：24 小時制 HH:MM
- 缺的欄位留空
- 編碼 UTF-8
- 學期開學日是 2025/9/1，學期結束日是 2025/12/19
- 「日期」範圍請涵蓋 2025/9/1 到 2025/12/19 之間所有有課的日期。
- 請仔細解析 PDF 中的表格，找出所有課程、時間和地點。
- 「當日提醒」與「學習建議」欄位先保持空白，後續會由系統再填入。
"""

    print("正在向 Gemini 提出請求，轉為 CSV…")
    response = model.generate_content([prompt_text, gemini_file])

    # 3. 刪除遠端檔案
    print("正在從 Gemini 雲端刪除暫存檔案...")
    genai.delete_file(gemini_file.name)
    print("✅ 暫存檔案已刪除。")

    if not response or not response.text:
        raise ValueError("❌ Gemini 沒有回傳內容，請檢查輸入或稍後再試。")

    # 4. 整理成純 CSV
    csv_output = response.text.strip()
    csv_output = csv_output.replace("```csv\n", "").replace("```", "")

    # 5. 存成本地檔
    with open("gemini_schedule.csv", "w", encoding="utf-8") as f:
        f.write(csv_output)
    print("✅ 已將 Gemini 產生的結果儲存為 'gemini_schedule.csv'")

    return csv_output

In [5]:
# =========================================================
# 3. CSV <-> DataFrame 與寫回 Google Sheet
# =========================================================
def csv_text_to_df(csv_text: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(csv_text))
    print("目前欄位：", list(df.columns))
    return df


def write_df_to_sheet(df: pd.DataFrame):
    ws.clear()
    set_with_dataframe(ws, df, include_index=False)
    print("✅ 已將資料寫入 Google Sheet『課表』工作表")


def read_df_from_sheet() -> pd.DataFrame:
    """
    從當前『課表』工作表讀資料，轉成 DataFrame。
    以第一列為欄名。
    """
    values = ws.get_all_values()
    if not values:
        raise ValueError("『課表』工作表目前是空的。")

    header = values[0]
    rows = values[1:]
    df = pd.DataFrame(rows, columns=header)
    print("從 Sheet 讀取到欄位：", list(df.columns))
    return df

In [6]:
# =========================================================
# 4. 國定假日處理 + AI 產生「當日提醒」與「學習建議」
# =========================================================

# ---------- 4-1. 國定假日處理 ----------
HOLIDAY_MAP = {
    (9, 29): "孔子誕辰紀念日",
    (10, 6): "中秋節",
    (10, 10): "國慶日",
    (10, 24): "台灣光復節",
}


def enforce_holidays(df: pd.DataFrame, log_fn=print) -> pd.DataFrame:
    """
    對 DataFrame 執行國定假日覆寫：
    - 若日期為指定的國定假日，則將「課程名稱」欄位改為「(節日)放假一天」
    - 不動其它欄位
    """

    if "日期" not in df.columns or "課程名稱" not in df.columns:
        log_fn("⚠️ 無法套用國定假日規則，缺少『日期』或『課程名稱』欄位。")
        return df

    changed_count = 0

    for idx, row in df.iterrows():
        date_str = str(row.get("日期", "")).strip()
        if not date_str:
            continue

        # 嘗試支援 "YYYY/MM/DD" 或 "YYYY-MM-DD"
        parsed_date = None
        for fmt in ("%Y/%m/%d", "%Y-%m-%d"):
            try:
                parsed_date = dt.datetime.strptime(date_str, fmt)
                break
            except ValueError:
                continue

        if not parsed_date:
            continue

        key = (parsed_date.month, parsed_date.day)
        if key in HOLIDAY_MAP:
            holiday_name = HOLIDAY_MAP[key]
            df.at[idx, "課程名稱"] = f"{holiday_name}放假一天"
            changed_count += 1

    if changed_count > 0:
        log_fn(f"🎌 已套用國定假日規則，共 {changed_count} 筆課程名稱改為『(節日)放假一天』。")
    else:
        log_fn("ℹ️ 本次資料中沒有落在國定假日的日期。")

    return df


# ---------- 4-2. AI 產生提醒與建議（以「課程名稱」為單位） ----------
def _call_gemini_for_course(row: pd.Series) -> tuple[str, str]:
    """
    針對「一門課」請 Gemini 產生：
    - reminder (當日提醒 / 行前準備)
    - advice   (學習建議)

    以「課程名稱 + 星期 + 典型時間」為基準，
    產生一組通用文字，套用到所有這門課的每一週。
    """

    course_name = str(row.get("課程名稱", "")).strip()
    weekday_str = str(row.get("星期", "")).strip()
    time_start = str(row.get("時間(起)", "")).strip()
    time_end = str(row.get("時間(迄)", "")).strip()
    location = str(row.get("地點", "")).strip()

    if not course_name:
        return "", ""

    prompt = f"""
你是一位大學生的行前提醒與學習規劃助理。

請針對下面這「一門課」產生兩行文字，內容要可以適用於這門課每一次上課（每一週通用），請不要加任何標題或多餘說明：

第 1 行：當日提醒（要攜帶的物品、服裝、注意事項等，約 15~30 個字，語氣自然）
第 2 行：學習建議（依照課程名稱與上課時間，給當天上課前後的學習建議，約 20~40 個字）

注意：
- 只能輸出兩行文字
- 不要加項目符號、JSON、解釋、前後空行

課程資訊：
- 課程名稱：{course_name}
- 星期：{weekday_str}
- 上課時間：{time_start} ~ {time_end}
- 上課地點：{location}
"""

    try:
        resp = model.generate_content(prompt)
        text = (resp.text or "").strip()

        # 取出非空行
        lines = [line.strip() for line in text.splitlines() if line.strip()]

        if not lines:
            return "", ""

        reminder = lines[0]
        advice = lines[1] if len(lines) > 1 else ""

        return reminder, advice

    except Exception as e:
        print(f"⚠️ 產生提醒/建議時發生錯誤（課程：{course_name}）：{e}")
        return "", ""


def add_ai_suggestions_to_df(df: pd.DataFrame, log_fn=print) -> pd.DataFrame:
    """
    針對 DataFrame：
    - 找出目前「當日提醒」「學習建議」為空的列
    - 以「課程名稱」分組，每一門課只 call Gemini 一次
    - 將同一門課的提醒/建議套用到所有相關列
    - 🚫 課程名稱包含「放假一天」的列會被跳過，不產生建議
    """

    # 確保欄位存在
    if "當日提醒" not in df.columns:
        df["當日提醒"] = ""
    if "學習建議" not in df.columns:
        df["學習建議"] = ""

    if "課程名稱" not in df.columns:
        log_fn("⚠️ DataFrame 裡找不到『課程名稱』欄位，無法產生建議。")
        return df

    def is_empty_cell(v) -> bool:
        return (pd.isna(v)) or (not str(v).strip())

    # 要處理的列：課程名稱有值、不是「放假一天」，且至少有一欄是空的
    indices_to_fill = []
    for idx, row in df.iterrows():
        course_name = str(row.get("課程名稱", "")).strip()

        # 沒課名或是放假的一律跳過
        if not course_name:
            continue
        if "放假一天" in course_name:
            continue

        if is_empty_cell(row.get("當日提醒", "")) or is_empty_cell(row.get("學習建議", "")):
            indices_to_fill.append(idx)

    if not indices_to_fill:
        log_fn("ℹ️ 所有課程的『當日提醒』『學習建議』都已經有內容，或都是放假日，略過 AI 產生。")
        return df

    need_df = df.loc[indices_to_fill]
    grouped = need_df.groupby("課程名稱")
    log_fn(f"🔎 共有 {len(need_df)} 筆列需要 AI 補完，屬於 {len(grouped)} 門課。")
    log_fn("👉 每門課只會呼叫一次 Gemini，避免超過 API 額度。")

    for course_name, group in grouped:
        log_fn(f"  ▶ 處理課程：{course_name}（共 {len(group)} 列）")

        rep_row = group.iloc[0]
        reminder, advice = _call_gemini_for_course(rep_row)

        if not reminder and not advice:
            log_fn(f"    ⚠️ 課程「{course_name}」未取得任何建議，略過。")
            continue

        for idx in group.index:
            if is_empty_cell(df.at[idx, "當日提醒"]):
                df.at[idx, "當日提醒"] = reminder
            if is_empty_cell(df.at[idx, "學習建議"]):
                df.at[idx, "學習建議"] = advice

    log_fn("✅ 已完成所有需要補完課程的『當日提醒』與『學習建議』產生。")
    return df

In [7]:
# =========================================================
# 5. 每日學術報導推薦（Phys.org RSS）
# =========================================================
PHYSORG_RSS_URL = "https://phys.org/rss-feed/science-news/"


def fetch_physorg_articles(limit: int = 20):
    """
    從 Phys.org RSS 抓最新幾篇學術/科學報導。
    回傳 list[dict]：{title, url, summary}
    """
    feed = feedparser.parse(PHYSORG_RSS_URL)

    articles = []
    for entry in feed.entries[:limit]:
        title = entry.get("title", "(no title)")
        link = entry.get("link", "")
        summary = entry.get("summary", "") or entry.get("description", "")

        # 移除 HTML tag
        summary_text = re.sub(r"<.*?>", "", summary).strip()

        articles.append({
            "title": title,
            "url": link,
            "summary": summary_text,
        })

    return articles


def get_daily_academic_article():
    """
    從 Phys.org 抓一批最新文章，依「今天日期」穩定地挑一篇推薦。
    回傳給 Gradio 顯示的 Markdown 字串。
    """
    articles = fetch_physorg_articles(limit=30)

    if not articles:
        return "⚠️ 暫時抓不到學術新聞，請稍後再試一次。"

    today = dt.date.today().isoformat()
    idx = hash(today) % len(articles)
    a = articles[idx]

    summary_short = a["summary"]
    if len(summary_short) > 200:
        summary_short = summary_short[:200] + "..."

    md = f"""
**{a['title']}**

{summary_short or '（原文無摘要，請直接點開連結閱讀）'}

🔗 [閱讀完整報導]({a['url']})
"""

    return md.strip()

In [8]:
# =========================================================
# 6. Gradio 後端函式：
#    A. 轉換 PDF ➜ 課表寫入 Sheet
#    B. 產生 AI 建議
#    C. 依日期查看課程提醒（含天氣 & 放假邏輯）
# =========================================================

def gradio_convert_pdf(pdf_path: str):
    """
    Tab 1 按鈕 1：
    - 上傳 PDF 課表
    - 呼叫 Gemini 轉 CSV
    - 套用國定假日規則（把當天課程名稱改成「(節日)放假一天」）
    - 不產生 AI 建議
    - 直接寫入 Google Sheet『課表』
    """
    logs = []

    def log(msg):
        print(msg)
        logs.append(msg)

    try:
        if not pdf_path:
            raise ValueError("請先上傳一個 PDF 檔案。")

        log(f"收到檔案路徑：{pdf_path}")

        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"找不到檔案：{pdf_path}")

        # 1. PDF ➜ CSV
        log("開始呼叫 Gemini 轉換課表成 CSV…")
        csv_text = pdf_path_to_csv_text(pdf_path)

        # 2. CSV ➜ DataFrame
        log("CSV 轉成 DataFrame…")
        df = csv_text_to_df(csv_text)

        # 3. 套用國定假日規則
        df = enforce_holidays(df, log_fn=log)

        # 4. 寫入 Google Sheet（暫時不含 AI 建議）
        log("寫入 Google Sheet『課表』工作表（已套用國定假日，尚未產生 AI 建議）…")
        write_df_to_sheet(df)

        log("✅ 完成：已寫入『課表』工作表。")

        return df, "\n".join(logs)

    except Exception as e:
        err_text = f"❌ 處理 PDF 時發生錯誤：{e}"
        tb = traceback.format_exc()
        print(err_text)
        print(tb)

        logs.append(err_text)
        logs.append("---- Traceback ----")
        logs.append(tb)

        return pd.DataFrame(), "\n".join(logs)


def gradio_generate_suggestions():
    """
    Tab 1 按鈕 2：
    - 從現有 Google Sheet『課表』讀資料
    - 先重新套用一次國定假日規則
    - 再只產生 / 更新『當日提醒』『學習建議』
    - 寫回同一個工作表
    """
    logs = []

    def log(msg):
        print(msg)
        logs.append(msg)

    try:
        log("從 Google Sheet『課表』讀取現有課表資料…")
        df = read_df_from_sheet()

        # 重新套用國定假日規則（確保就算有人手動改掉也會被改回）
        df = enforce_holidays(df, log_fn=log)

        log("開始根據課程名稱與時間，AI 產生／更新『當日提醒』『學習建議』…")
        df = add_ai_suggestions_to_df(df, log_fn=log)

        log("寫回 Google Sheet『課表』工作表（包含 AI 產生的欄位）…")
        write_df_to_sheet(df)

        log("✅ 完成：已更新『課表』中的「當日提醒」「學習建議」。")

        return df, "\n".join(logs)

    except Exception as e:
        err_text = f"❌ 產生提醒/建議時發生錯誤：{e}"
        tb = traceback.format_exc()
        print(err_text)
        print(tb)

        logs.append(err_text)
        logs.append("---- Traceback ----")
        logs.append(tb)

        return pd.DataFrame(), "\n".join(logs)


def df_to_html_table(df: pd.DataFrame) -> str:
    """把 DataFrame 轉成普通 HTML 表格，方便用 gr.HTML 顯示。"""
    if df is None or df.empty:
        return "<p>目前沒有可顯示的資料。</p>"

    return df.to_html(
        index=False,
        border=0,
        classes="df-plain-table"
    )


def gradio_convert_pdf_ui(pdf_path: str):
    """
    UI 包裝版本：PDF ➜ DataFrame ➜ HTML 表格。
    成功時只回傳一行簡短訊息；出錯時才顯示完整 log。
    """
    df, logs = gradio_convert_pdf(pdf_path)  # 你原本的主流程
    html = df_to_html_table(df)

    # 如果有錯誤（含「❌」），就把完整 logs 顯示出來
    if "❌" in logs:
        ui_msg = logs
    else:
        ui_msg = "✅ 已成功將 PDF 轉成課表並寫入 Google Sheet。"

    return html, ui_msg


def gradio_generate_suggestions_ui():
    """
    UI 包裝版本：產生建議 ➜ DataFrame ➜ HTML 表格。
    成功時只回傳一行簡短訊息；出錯時才顯示完整 log。
    """
    df, logs = gradio_generate_suggestions()  # 你原本的主流程
    html = df_to_html_table(df)

    if "❌" in logs:
        ui_msg = logs
    else:
        ui_msg = "✅ 已完成 AI 當日提醒與學習建議更新。"

    return html, ui_msg
def gradio_view_reminders(date_value: str):
    """
    Tab2：「📅 查看當日課程提醒」
    1. 從 Google Sheet『課表』讀取當天的所有課程
    2. 組成課程提醒小卡（Markdown）
    3. 最上方加上一段「今日天氣」說明（用 OpenWeather）
    4. 放假當天：若全部課程名稱都含「放假一天」，只顯示一張「(節日)放假一天！」卡片
    """
    try:
        # ---- 1. 決定要看的日期字串 ----
        if date_value is None or str(date_value).strip() == "":
            date_str = dt.date.today().strftime("%Y/%m/%d")
        else:
            date_str = str(date_value).strip()

        # ---- 2. 從 Google Sheet 讀出整張課表 ----
        records = ws.get_all_records()
        if not records:
            return "⚠️ 課表工作表目前是空的，請先在 Tab1 上傳並產生課表。"

        df = pd.DataFrame(records)

        if "日期" not in df.columns:
            return "⚠️ 找不到「日期」欄位，請確認課表格式是否正確。"

        # ---- 3. 篩選當天的課程 ----
        sub = df[df["日期"] == date_str]

        # 先準備天氣說明（台北可改成你自己的城市）
        weather_note = build_weather_note(city="Taipei", country_code="TW")

        # ---- 3-1. 當天沒有課程（完全沒有出現在課表）----
        if sub.empty:
            base_text = f"### {date_str} 今日沒有課程\n\n今日沒有課程，加油寫作業與好好休息吧！"
            if weather_note:
                return base_text + "\n\n---\n\n" + weather_note
            else:
                return base_text

        # ---- 3-2. 判斷是否為「放假一天」的情況 ----
        if "課程名稱" in sub.columns:
            names = [str(x).strip() for x in sub["課程名稱"] if str(x).strip()]
        else:
            names = []

        is_holiday_day = bool(
            names and all("放假一天" in name for name in names)
        )

        if is_holiday_day:
            # 取第一個課程名稱，抽出節日名稱
            first = names[0]
            # 例如 "中秋節放假一天" → "中秋節放假一天！"
            if first.endswith("放假一天"):
                holiday_text = first.replace("放假一天", "放假一天！")
            else:
                holiday_text = first + "！"

            holiday_md = f"### {date_str} {holiday_text}"

            if weather_note:
                return weather_note + "\n\n---\n\n" + holiday_md
            else:
                return holiday_md

        # ---- 3-3. 一般有課日：組課程小卡 ----
        cards = []
        for _, row in sub.iterrows():
            course = str(row.get("課程名稱", "")).strip()
            start = str(row.get("時間(起)", "")).strip()
            end = str(row.get("時間(迄)", "")).strip()
            place = str(row.get("地點", "")).strip()
            remind = str(row.get("當日提醒", "")).strip()
            sugg = str(row.get("學習建議", "")).strip()

            if not course:
                course = "（未命名課程）"

            lines = [f"### {course}"]

            if start or end:
                lines.append(f"- 🕒 時間：{start} ~ {end}")
            if place:
                lines.append(f"- 📍 地點：{place}")
            if remind:
                lines.append(f"- 📌 當日提醒：{remind}")
            if sugg:
                lines.append(f"- 📖 學習建議：{sugg}")

            cards.append("\n".join(lines))

        cards_text = "\n\n---\n\n".join(cards)

        # ---- 4. 把天氣放在最上方（有拿到才顯示）----
        if weather_note:
            return weather_note + "\n\n---\n\n" + cards_text
        else:
            return cards_text

    except Exception as e:
        return f"❌ 讀取課表或產生提醒時發生錯誤：{e}"


In [9]:
import datetime as dt
import gspread

# ========================================
# 🗂️ 留言板設定
# ========================================
COMMENT_SHEET_NAME = "留言板"  # ← Google Sheet 分頁名稱

def comment_now_str():
    """取得現在時間 YYYY/MM/DD HH:MM"""
    return dt.datetime.now().strftime("%Y/%m/%d %H:%M")


# ========================================
# 🗂️ Google Sheet 相關
# ========================================

def get_comment_ws():
    """取得「留言板」分頁，沒有就自動建立（用前面已經開好的 sh）"""
    global sh   # 🔸 告訴 Python 要用前面那個 sh
    try:
        ws = sh.worksheet(COMMENT_SHEET_NAME)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=COMMENT_SHEET_NAME, rows=2000, cols=5)
        # 🔸 這裡標題列用『時間 / 心情 / 留言』
        ws.update("A1:C1", [["時間", "心情", "留言"]])
    return ws


def load_comments_from_sheet():
    """從 Google Sheet 載入留言為 list(dict)"""
    ws = get_comment_ws()
    rows = ws.get_all_records()

    cards = []
    for r in rows:
        cards.append({
            "time": r.get("時間", ""),
            "nickname": r.get("心情", ""),  # 🔸 對應上面標題「心情」
            "message": r.get("留言", "")
        })

    cards.reverse()  # 保持你原本的最新在最上面
    return cards


def append_comment_to_sheet(card):
    """將留言新增到 Google Sheet"""
    ws = get_comment_ws()
    ws.append_row([card["time"], card["nickname"], card["message"]])


# ========================================
# 💬 HTML 渲染：留言牆畫面
# ========================================

def comment_render_cards(cards):
    """把留言轉成 HTML 顯示"""
    if not cards:
        return "還沒有任何留言！"

    html = ""
    for c in cards:
        html += f"""
        <div style="
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 8px;
            margin-bottom: 8px;
        ">
            <div style="font-size:12px; color:#555;">
                <b>{c['nickname']}</b>　<span>{c['time']}</span>
            </div>
            <div style="margin-top:4px; font-size:14px; white-space:pre-wrap;">
                {c['message']}
            </div>
        </div>
        """
    return html


# ========================================
# 🚀 初始化：從 Sheet 載入留言
# ========================================
def comment_init_from_sheet():
    cards = load_comments_from_sheet()
    return cards, comment_render_cards(cards)


# ========================================
# ➕ 新增留言（寫入 Google Sheet）
# ========================================
def comment_add_to_sheet(nickname, message, cards):
    nickname = (nickname or "").strip()
    message = (message or "").strip()

    if not nickname or not message:
        # 沒寫好 → 不新增 → 重新渲染即可
        return cards, comment_render_cards(cards), nickname, message

    # 建立留言
    new_card = {
        "time": comment_now_str(),
        "nickname": nickname,
        "message": message,
    }

    # 寫入 Google Sheet
    append_comment_to_sheet(new_card)

    # 重新從 Google Sheet 抓最新留言
    cards = load_comments_from_sheet()

    # 回傳給 Gradio
    return cards, comment_render_cards(cards), "", ""   # 清空暱稱與留言


In [13]:
# =========================================================
# 7. Gradio 介面 + CSS（乾淨版 UI）
# =========================================================
CUSTOM_CSS = """
.gradio-container {
    background: #f5f7fb;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}

/* 主要內容寬度限制 */
main {
    max-width: 1080px;
    margin: 0 auto !important;
}

/* 頁首標題區 */
.app-header {
    text-align: left;
    padding: 1.5rem 0 0.75rem 0;
}
.app-title {
    font-size: 1.6rem;
    font-weight: 700;
    color: #1f2933;
    margin-bottom: 0.25rem;
}
.app-subtitle {
    font-size: 0.95rem;
    color: #6b7280;
}

/* 卡片外觀 */
.card {
    background: #ffffff;
    border-radius: 0px;
    padding: 1.25rem 1.5rem;
    box-shadow: 0 10px 30px rgba(15, 23, 42, 0.06);
    border: 1px solid #e5e7eb;
}

/* 小標題 */
.card-title {
    font-size: 1.05rem;
    font-weight: 600;
    color: #111827;
    margin-bottom: 0.35rem;
}
.card-desc {
    font-size: 0.9rem;
    color: #6b7280;
    margin-bottom: 0.75rem;
}

/* 按鈕 */
button {
    border-radius: 999px !important;
    font-weight: 600 !important;
}

/* 分隔線 */
.section-divider {
    margin: 1.5rem 0 0.75rem 0;
    border-top: 1px dashed #e5e7eb;
}

/* Placeholder Markdown */
.reminder-placeholder {
    color: #9ca3af;
    font-size: 0.9rem;
}

/* ===== HTML 表格樣式（直角、非膠囊） ===== */
.df-plain-table {
    border-collapse: collapse;
    width: 100%;
    font-size: 0.9rem;
}

.df-plain-table th,
.df-plain-table td {
    border: 1px solid #d1d5db;
    padding: 6px 8px;
    text-align: left;
    white-space: nowrap;
}

.df-plain-table th {
    background: #f3f4f6;
    font-weight: 600;
}

.df-plain-table tbody tr:nth-child(even) td {
    background: #f9fafb;
}
/* 固定高度 + 可捲動的預覽區塊 */
.scroll-box {
    max-height: 320px;     /* 🔧 想高一點就調這裡 */
    overflow-y: auto;
    padding-right: 4px;
}

/* 可選：縮細卷軸，讓畫面好看一點 */
.scroll-box::-webkit-scrollbar {
    width: 6px;
}
.scroll-box::-webkit-scrollbar-thumb {
    background: #d1d5db;
    border-radius: 999px;
}
.scroll-box::-webkit-scrollbar-track {
    background: transparent;
}

"""
def show_processing_convert():
    return "⏳ 正在處理 PDF，請稍候..."

def show_processing_ai():
    return "⏳ 正在產生 AI 建議，請稍候..."

with gr.Blocks(css=CUSTOM_CSS) as demo:
    # ========= 頁首 =========
    with gr.Column(elem_classes="app-header"):
        gr.Markdown(
            """
<div class="app-title">📚 課程小助理</div>
<div class="app-subtitle">
讓我來幫助你整理課表，當你的貼心助理吧！
</div>
            """
        )

    # ========= Tabs =========
    with gr.Tabs():
        # ---------- Tab 1 ----------
        with gr.Tab("① 上傳課表與生成建議"):
            with gr.Column(elem_classes="card"):
                gr.Markdown(
                    """
<div class="card-title">步驟 1：上傳 PDF 課表並寫入 Google Sheet</div>
<div class="card-desc">
如果已上傳過則可略過此步驟。
</div>
                    """
                )
                with gr.Row():
                    pdf_input = gr.File(label="📄 上傳課表 PDF", type="filepath")
                    btn_convert = gr.Button("轉換", scale=0)

                # 🔹 改成 HTML 表格預覽（不用 gr.Dataframe 了）
                table_out1 = gr.HTML(elem_classes="scroll-box")
                log_out1 = gr.Markdown(label="處理狀態 / 訊息")

                # 先顯示處理中，再真正執行轉換
                btn_convert.click(
                    fn=show_processing_convert,
                    inputs=None,
                    outputs=log_out1,  # 先只更新「處理狀態 / 訊息」
                ).then(
                    fn=gradio_convert_pdf_ui,        # 再跑真正的轉換
                    inputs=pdf_input,
                    outputs=[table_out1, log_out1],  # 表格 + 最終訊息
                )

            gr.HTML('<div class="section-divider"></div>')

            with gr.Column(elem_classes="card"):
                gr.Markdown(
                    """
<div class="card-title">步驟 2：生成 AI 建議</div>
<div class="card-desc">
依課程推論當日提醒、學習建議並寫入 Google Sheet。
</div>
                    """
                )

                btn_ai = gr.Button("2️⃣ 生成 / 更新 AI 建議")

                # 🔹 一樣用 HTML 表格預覽
                table_out2 = gr.HTML(elem_classes="scroll-box")
                log_out2 = gr.Markdown(label="AI 建議生成狀態")

                # 先顯示處理中，再真正執行 AI 建議
                btn_ai.click(
                    fn=show_processing_ai,
                    inputs=None,
                    outputs=log_out2,  # 先顯示「正在產生 AI 建議」
                ).then(
                    fn=gradio_generate_suggestions_ui,  # 再真正去跑 AI
                    inputs=[],
                    outputs=[table_out2, log_out2],
                )

        # ---------- Tab 2 ----------
        with gr.Tab("② 課程提醒與文章推薦"):
            # 課程提醒卡片
            with gr.Column(elem_classes="card"):
                gr.Markdown(
                    """
<div class="card-title">📅 課程提醒小卡</div>
<div class="card-desc">
請輸入想查看的日期（預設為當日日期），會附上即時天氣。
</div>
                    """
                )

                with gr.Row():
                    with gr.Column(scale=1):
                        today_str = dt.date.today().strftime("%Y/%m/%d")
                        date_input = gr.Textbox(
                            label="選擇日期（YYYY/MM/DD）",
                            value=today_str,
                        )
                        btn_view = gr.Button("🔔 查看當日課程提醒")

                    with gr.Column(scale=2):
                        cards_md = gr.Markdown(
                            value="<div class='reminder-placeholder'>請選擇日期後點擊按鈕。</div>",
                             elem_classes="scroll-box",
                        )

                btn_view.click(
                    fn=gradio_view_reminders,
                    inputs=date_input,
                    outputs=cards_md,
                )

            gr.HTML('<div class="section-divider"></div>')

            # 每日學術報導推薦卡片
            with gr.Column(elem_classes="card"):
                gr.Markdown(
                    """
<div class="card-title">🎓 每日學術報導推薦</div>
<div class="card-desc">
從 Phys.org 抽一則適合今日閱讀的學術 / 科普文章。
</div>
                    """
                )

                btn_article = gr.Button("查詢 🔍")
                article_md = gr.Markdown(
                    value="<span class='reminder-placeholder'>按下按鈕以推薦文章。</span>"
                )

                btn_article.click(
                    fn=get_daily_academic_article,
                    inputs=[],
                    outputs=article_md,
                )

        # ---------- Tab 3：留言小卡牆 ----------
        with gr.Tab("③ 留言小卡牆"):
            with gr.Column(elem_classes="card"):
                gr.Markdown(
                    """
<div class="card-title">🐣 留言小卡牆</div>
<div class="card-desc">
留下今天的心情、小提醒或給未來自己的話吧！
</div>
                    """
                )

                # 🟡 初始化時就從 Sheet 載入留言
                initial_cards = load_comments_from_sheet()
                comment_state = gr.State(initial_cards)

                with gr.Row():
                    # 左邊：輸入區
                    with gr.Column(scale=1):
                        comment_nickname = gr.Textbox(
                            label="心情",
                            placeholder="輸入今日心情"
                        )
                        comment_message = gr.Textbox(
                            label="留言內容",
                            placeholder="寫下想說的話...",
                            lines=4
                        )
                        comment_submit_btn = gr.Button("送出留言")

                    # 右邊：留言牆 + 重新載入按鈕
                    with gr.Column(scale=1):
                        comment_cards_html = gr.HTML(
                            label="留言牆",
                            value=comment_render_cards(initial_cards)
                        )
                        reload_btn = gr.Button("🔁 顯示 / 重新載入之前的留言")

                # ✅ 送出留言：寫進 Google Sheet 再重讀
                def comment_add_no_delete(nickname, message, cards):
                    if cards is None:
                        cards = []

                    nickname = (nickname or "").strip()
                    message = (message or "").strip()

                    # 沒填完整就不新增，只重畫畫面
                    if not nickname or not message:
                        return (
                            cards,
                            comment_render_cards(cards),
                            nickname,
                            message,
                        )

                    new_card = {
                        "nickname": nickname,
                        "message": message,
                        "time": comment_now_str(),
                    }

                    append_comment_to_sheet(new_card)
                    cards = load_comments_from_sheet()

                    return (
                        cards,
                        comment_render_cards(cards),
                        "",   # 清空「心情」
                        "",   # 清空留言
                    )

                # 🔁 顯示 / 重新載入過去留言
                def reload_comments(cards):
                    cards = load_comments_from_sheet()
                    return cards, comment_render_cards(cards)

                comment_submit_btn.click(
                    comment_add_no_delete,
                    inputs=[comment_nickname, comment_message, comment_state],
                    outputs=[comment_state, comment_cards_html, comment_nickname, comment_message]
                )

                reload_btn.click(
                    reload_comments,
                    inputs=comment_state,
                    outputs=[comment_state, comment_cards_html]
                )



demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5b4d1a06c4a07b0206.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


從 Google Sheet『課表』讀取現有課表資料…
從 Sheet 讀取到欄位： ['日期', '星期', '課程名稱', '時間(起)', '時間(迄)', '地點', '當日提醒', '學習建議']
🎌 已套用國定假日規則，共 16 筆課程名稱改為『(節日)放假一天』。
開始根據課程名稱與時間，AI 產生／更新『當日提醒』『學習建議』…
🔎 共有 176 筆列需要 AI 補完，屬於 10 門課。
👉 每門課只會呼叫一次 Gemini，避免超過 API 額度。
  ▶ 處理課程：3D列印與創新應用（共 16 列）
  ▶ 處理課程：作業系統（共 14 列）
  ▶ 處理課程：微積分乙(一)（共 16 列）
  ▶ 處理課程：機率論（共 28 列）
  ▶ 處理課程：環境與傳播（共 14 列）
  ▶ 處理課程：程式語言（共 16 列）
  ▶ 處理課程：網際網路概論（共 16 列）
  ▶ 處理課程：網際網路程式設計（共 14 列）
  ▶ 處理課程：線性代數（共 28 列）
  ▶ 處理課程：體育(匹克球)（共 14 列）
✅ 已完成所有需要補完課程的『當日提醒』與『學習建議』產生。
寫回 Google Sheet『課表』工作表（包含 AI 產生的欄位）…
✅ 已將資料寫入 Google Sheet『課表』工作表
✅ 完成：已更新『課表』中的「當日提醒」「學習建議」。
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5b4d1a06c4a07b0206.gradio.live
